# Trajectory adapter on LTX-Video 2B
Run top to bottom on a Colab GPU (A100 or L4). Settings: `configs/config.yaml`. Checkpoints go to Drive.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import os
from google.colab import drive
if not os.path.isdir('/content/drive/MyDrive'):
    drive.mount('/content/drive')

In [ ]:
# code + data zip made from the repo (data/ included); change the name if yours differs
ZIP = '/content/drive/MyDrive/ie643_full.zip'
!unzip -o -q {ZIP} -d /content/ie643
%cd /content/ie643
!pip install -q -r requirements.txt
!python -m pytest tests -q

In [ ]:
# optional: Hugging Face login (the token is typed, never printed or saved)
import getpass
from huggingface_hub import login
login(getpass.getpass('HF token (empty to skip): ') or None)

## Train (about 12 min per epoch on an A100)

In [ ]:
RUN = 'classic'          # new name for every experiment
CK = f'/content/drive/MyDrive/ie643_ckpt/{RUN}'
os.makedirs(CK, exist_ok=True)
!python -u scripts/train.py --save_dir {CK} 2>&1 | tee -a {CK}/train_stdout.log

In [ ]:
# did the adapter learn to use the trajectory? val_obj_gain above 0 is good, val_bg_gain should stay near 0
import json
for l in open(f'{CK}/log.jsonl'):
    r = json.loads(l)
    print(r['epoch'], 'obj_gain %.5f  bg_gain %.5f  inject %.4f' % (r['val_obj_gain'], r['val_bg_gain'], r['inject_ratio']))

## Look at the result
`as3` = adapter at 3x its trained strength; `ref0.8+comp2.5` = training-free drag-video start; `rel0`, `mg3`, `until0.4` are further variants.

In [ ]:
!python scripts/demo_grid.py --checkpoint {CK}/best.pt --num 3 --variants base,adapter,adapter+as3,adapter+as6,ref0.8+comp2.5 --out {CK}/grid

In [ ]:
# one example traced from input to metrics
!python scripts/trace_example.py --checkpoint {CK}/best.pt --direction right --adapter_scale 3 --out {CK}/trace

## Benchmark: plain LTX vs the adapter (8 clips x 5 paths, matched seeds, about 25 min)

In [ ]:
!python scripts/benchmark.py --checkpoint {CK}/best.pt --num_images 8 --adapter_scale 3 --out {CK}/bench 2>&1 | grep --line-buffered -v -i -E "warning|onednn|absl|cudart|it/s"
print(open(f'{CK}/bench/summary.md').read())

## Optional
- Out-of-domain images: put clean single-object jpg/png files in a Drive folder and add `--ood_dir <folder>` to the benchmark.
- Focus-loss ablation: train again with `--focus_loss 0`, benchmark both with the same `--seeds`.
- Demo app: `!pip install -q gradio` then `!python app/app.py --checkpoint {CK}/best.pt --share`.